[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter7/function-agent-llamaindex.ipynb)

# Chapter 7: Research & Planner Agent with LlamaIndex FunctionAgent

This notebook demonstrates building a multi-tool agent using LlamaIndex's FunctionAgent with web search, RAG, and calculator tools powered by Anthropic's Claude.

**What you'll learn:**
- Build a FunctionAgent with multiple tools (web search, RAG, calculator)
- Use Tavily for real-time web search and LlamaIndex for PDF-based RAG
- Stream agent events including tool calls, results, and final outputs

**Prerequisites:** `pip install llama-index tavily-python python-dotenv` and set `ANTHROPIC_API_KEY` and `TAVILY_API_KEY`.

## Setup

We import LlamaIndex's agent framework, Anthropic LLM integration, and Tavily for web search, then load API keys from the environment.

In [1]:
import os
import asyncio
from typing import Optional

from llama_index.llms.anthropic import Anthropic

from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.agent.workflow import FunctionAgent, AgentWorkflow
from llama_index.core.workflow import Context
from llama_index.core.tools import FunctionTool, QueryEngineTool
from llama_index.core.callbacks import CallbackManager, LlamaDebugHandler

from llama_index.core.agent.workflow import (
    ToolCall, ToolCallResult, AgentInput, AgentOutput, AgentStream
)
from llama_index.core.workflow import Workflow

from tavily import AsyncTavilyClient

from dotenv import load_dotenv
load_dotenv()

False

In [2]:
#!pip install llama-index==0.14.14 llama-index-llms-anthropic anthropic -U --quiet

## Build RAG Tool

We define three tools the agent can call: a web search tool for live information, a calculator for arithmetic, and a RAG tool that indexes the GPT-2 paper for domain-specific questions.

In [3]:
# define the LLM
model_name = "claude-sonnet-4-5"
llm = Anthropic(model=model_name)
Settings.llm = llm

# define tools: Tavily tool, calculator tool
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

async def web_search(query: str) -> str:
    """Use the web to get up-to-date information and sources."""
    client = AsyncTavilyClient(api_key=TAVILY_API_KEY)
    result = await client.search(query, search_depth="advanced", include_raw_content=False)
    return str(result)

web_search_tool = FunctionTool.from_defaults(
    fn=web_search,
    name="web_search",
    description="Search the web for fresh information and return relevant results with links.",
)

def calc(expression: str) -> str:
    """Safely evaluate simple arithmetic expressions like '2 + 2*5'."""
    # Very minimal "safe eval" for demo purposes; expand as needed.
    allowed = set("0123456789+-*/(). %")
    if not set(expression) <= allowed:
        return "Error: Unsupported characters."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

calc_tool = FunctionTool.from_defaults(
    fn=calc,
    name="calculator",
    description="Evaluate basic arithmetic expressions, e.g., '12.5 * 3 - 7'.",
)

# define RAG tool
def build_rag_tool(file: str = "language_models_are_unsupervised_multitask_learners.pdf") -> Optional[QueryEngineTool]:
    try:
        docs = SimpleDirectoryReader(input_files=[file]).load_data()
        if not docs:
            return None
        index = VectorStoreIndex.from_documents(docs)
        qengine = index.as_query_engine(similarity_top_k=5)
        return QueryEngineTool.from_defaults(
            query_engine=qengine,
            name="gpt2-paper-tool",
            description=(
                "Answer questions from the GPT-2 paper"
            ),
            return_direct=False,
        )
    except Exception as e:
        print(f"DEBUG EXCeption {e}")
        return None

rag_tool = build_rag_tool()

tools = [web_search_tool, calc_tool, rag_tool]

2026-02-15 10:37:22,911 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


## Create Agent

The FunctionAgent wraps the LLM with a system prompt and tool list, enabling it to autonomously decide which tools to call for each query.

In [4]:
# Build the agent
agent = FunctionAgent(
    tools=tools,
    llm=llm,
    system_prompt=(
        "You are a research & planning assistant. "
        "Use tools when helpful. Prefer rag_tool for questions about GPT-2 "
        "When using web_search, summarize concisely and include source links. "
        "Show brief, actionable plans."
    ),
)

## Query the Agent

We send queries to the agent and stream its reasoning steps — tool calls, results, and the final synthesized answer — to observe the full decision loop.

In [5]:
ctx = Context(agent)
q1 = (
    "I'm planning a day trip to Santa Cruz this Saturday. "
    "Find 2-3 must-do activities (with links), estimate total ticket costs for two adults "
    "if needed, and suggest a 6-hour itinerary. Keep it tight."
)

# 1) Kick off the workflow (returns a handler)
handler = agent.run(user_msg=q1, ctx=ctx)

# 2) Stream events as they happen (incl. ToolCall/ToolCallResult)
async for event in handler.stream_events():
    if isinstance(event, AgentInput):
        print(f"\n🟦 AgentInput from {event.current_agent_name}:\n{event.input}")
    elif isinstance(event, AgentStream):
        # token-level model deltas
        print(event.delta, end="", flush=True)
    elif isinstance(event, ToolCall):
        print(f"\n🔧 ToolCall: {event.tool_name}  args={event}")
    elif isinstance(event, ToolCallResult):
        print(f"📥 ToolResult ({event.tool_name}): {str(event.tool_output)[:1000]}...")
    elif isinstance(event, AgentOutput):
        print(f"\n✅ Final from {event.current_agent_name}:\n{event.response.content}\n")



🟦 AgentInput from Agent:
[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are a research & planning assistant. Use tools when helpful. Prefer rag_tool for questions about GPT-2 When using web_search, summarize concisely and include source links. Show brief, actionable plans.')]), ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I'm planning a day trip to Santa Cruz this Saturday. Find 2-3 must-do activities (with links), estimate total ticket costs for two adults if needed, and suggest a 6-hour itinerary. Keep it tight.")])]


2026-02-15 10:37:24,447 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


I'll help you plan your Santa Cruz day trip. Let me search for the best activities and current information.
✅ Final from Agent:
I'll help you plan your Santa Cruz day trip. Let me search for the best activities and current information.


🔧 ToolCall: web_search  args=tool_name='web_search' tool_kwargs={'query': 'Santa Cruz must do activities attractions 2024'} tool_id='toolu_01Kx4Cte74gpdgyuHLvhSLfw'

🔧 ToolCall: web_search  args=tool_name='web_search' tool_kwargs={'query': 'Santa Cruz Beach Boardwalk ticket prices 2024'} tool_id='toolu_01WDVnwsMB6DJzhjV4kKSemE'


2026-02-15 10:37:29,461 - INFO - HTTP Request: POST https://api.tavily.com/search "HTTP/1.1 200 OK"


📥 ToolResult (web_search): {'query': 'Santa Cruz Beach Boardwalk ticket prices 2024', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://beachboardwalk.com/tickets/', 'title': 'Santa Cruz Beach Boardwalk Ride Tickets and Season Pass Info', 'content': "Skip to content\n\nCalifornia's Largest Admission Free Amusement Park\n\n \n\n## 2026 Season Passes\n\nChoose from three options.  \nBuy online & save up to $20!\n\n(Valid through 12/31/26)\n\nStarting at $104.95 + tax\n\nBuy Now\n\nBuy As Gift\n\n## Ride Wristbands\n\nUnlimited rides all day long.  \nBuy online and save up to 20%\\\n\n(\\Purchase by midnight the day before your visit to save!)\n\nStarting at $44.95 + tax\n\nBuy Now\n\n## MyBoardwalk Cards\n\nValid for individual rides, attractions, games, arcades, food and retail.\n\n(Bonus points not valid for food or retail)\n\n$50, $100, and $200\n\nBuy Now\n\n## Group Tickets\n\nOptions for 15 or more.\n\nMore Info\n\n## Event Tickets\n\nBrowse ti

...

📥 ToolResult (web_search): {'query': 'Santa Cruz must do activities attractions 2024', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://bridgesandballoons.com/fun-things-to-do-in-santa-cruz-california/', 'title': '34 fun things to do in Santa Cruz, California - Bridges & Balloons', 'content': '### 16. Visit the Mystery Spot\n\nThe Mystery Spot calls itself a gravitational anomaly where the laws of physics go haywire. Others will call it a tourist trap. Either way, the lighthearted optical illusion attraction can be a fun way to spend an hour, taking improbable photos of your friends in mid-air.\n\nBuy tickets in advance online to avoid disappointment. It’s a really popular place!\n\n### 17. Go for a coastal hike\n\nThere are some beautiful coastal hikes in Santa Cruz. One favourite is the Old Cove Landing Trail at Wilder Ranch State Park which takes you on a magnificent 7.5 mile coastal track. If you’re lucky, you’ll spot dolphins, seals and humpb

2026-02-15 10:37:31,237 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



✅ Final from Agent:
None


🔧 ToolCall: calculator  args=tool_name='calculator' tool_kwargs={'expression': '44.95 * 2'} tool_id='toolu_01Dco17bVYd45B5opSLVvDuG'
📥 ToolResult (calculator): 89.9...

🟦 AgentInput from Agent:
[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are a research & planning assistant. Use tools when helpful. Prefer rag_tool for questions about GPT-2 When using web_search, summarize concisely and include source links. Show brief, actionable plans.')]), ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I'm planning a day trip to Santa Cruz this Saturday. Find 2-3 must-do activities (with links), estimate total ticket costs for two adults if needed, and suggest a 6-hour itinerary. Keep it tight.")]), ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'usage': {'input_tokens': 814, 'output_tokens': 131}, 'stop_reason'

2026-02-15 10:37:32,880 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


## Santa Cruz Day Trip Plan

### **Top 3 Must-Do Activities:**

1. **Santa Cruz Beach Boardwalk** - California's oldest amusement park with iconic Giant Dipper roller coaster (1924)
   - Link: https://beachboardwalk.com/
   
2. **Natural Bridges State Beach** - Stunning coastal views, tide pools, and beach walks
   - Link: https://www.santacruz.org/
   
3. **Santa Cruz Wharf** - Historic pier with restaurants, sea lions, and ocean views
   - Link: https://www.santacruz.org/

### **Estimated Costs for 2 Adults:**
- **Boardwalk unlimited ride wristbands**: ~$90 (buy online in advance for $44.95/person + tax)
- **Parking**: ~$15-20
- **Natural Bridges/Wharf**: FREE (state park may have $10 parking)
- **Total**: ~$100-120

### **6-Hour Itinerary:**

**10:00 AM** - Start at **Natural Bridges State Beach** (1 hour)  
Explore tide pools, walk the beach, photo ops

**11:15 AM** - Drive to **Santa Cruz Beach Boardwalk** (10 min)  
**11:30 AM - 2:30 PM** - Boardwalk fun (3 hours)  
Ride Giant Di

In [6]:
ctx = Context(agent)
q2 = "What are the key takeaways of the GPT-2 paper? Summarize in 5 bullets."

# 1) Kick off the workflow (returns a handler)
handler = agent.run(user_msg=q2, ctx=ctx)

# 2) Stream events as they happen (incl. ToolCall/ToolCallResult)
async for event in handler.stream_events():
    if isinstance(event, AgentInput):
        print(f"\n🟦 AgentInput from {event.current_agent_name}:\n{event.input}")
    elif isinstance(event, AgentStream):
        # token-level model deltas
        print(event.delta, end="", flush=True)
    elif isinstance(event, ToolCall):
        print(f"\n🔧 ToolCall: {event.tool_name}  args={event}")
    elif isinstance(event, ToolCallResult):
        print(f"📥 ToolResult ({event.tool_name}): {str(event.tool_output)[:1000]}...")
    elif isinstance(event, AgentOutput):
        print(f"\n✅ Final from {event.current_agent_name}:\n{event.response.content}\n")



🟦 AgentInput from Agent:
[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are a research & planning assistant. Use tools when helpful. Prefer rag_tool for questions about GPT-2 When using web_search, summarize concisely and include source links. Show brief, actionable plans.')]), ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='What are the key takeaways of the GPT-2 paper? Summarize in 5 bullets.')])]


2026-02-15 10:37:43,492 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



✅ Final from Agent:
None


🔧 ToolCall: gpt2-paper-tool  args=tool_name='gpt2-paper-tool' tool_kwargs={'input': 'What are the key takeaways and main contributions of the GPT-2 paper?'} tool_id='toolu_01APxQz2hCJv57P3QpkpEXug'


2026-02-15 10:37:44,737 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-15 10:37:51,995 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


📥 ToolResult (gpt2-paper-tool): Based on the document, the key takeaways and main contributions of the GPT-2 paper are:

1. **Zero-shot multitask learning**: GPT-2 demonstrates that a large language model trained on a sufficiently large and diverse dataset can perform well across many domains and datasets without explicit supervision. It achieves state-of-the-art performance on 7 out of 8 tested language modeling datasets in a zero-shot setting.

2. **Task diversity without fine-tuning**: The model shows the ability to perform a surprising variety of tasks in zero-shot settings, including:
   - Reading comprehension (CoQA dataset - 55 F1)
   - Question answering
   - Summarization (CNN and Daily Mail dataset)
   - Translation (WMT-14 English-French)
   - Winograd Schema Challenge (70.70% accuracy, improving state-of-the-art by 7%)

3. **Scale matters**: The research demonstrates that high-capacity models trained to maximize the likelihood of a sufficiently varied text corpus can learn 

2026-02-15 10:37:53,118 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


## Key Takeaways of the GPT-2 Paper

• **Zero-shot multitask learning works**: GPT-2 demonstrates that large language models trained on diverse text can perform well across many tasks without task-specific training or fine-tuning, achieving state-of-the-art on 7 out of 8 language modeling datasets

• **Scale and diversity are critical**: The research shows that model capacity combined with training on sufficiently large and varied text corpora enables emergent task performance without explicit supervision

• **Natural language as task interface**: The model can perform different tasks (summarization, translation, Q&A) by responding to natural language prompts and context cues (e.g., "TL;DR:" triggers summarization behavior)

• **Impressive breadth of capabilities**: GPT-2 handles diverse tasks including reading comprehension (55 F1 on CoQA), translation, and notably improves Winograd Schema Challenge performance by 7% to reach 70.70% accuracy

• **Promising but not production-ready**: 